# BD Plate Reader: finish (export and evaluate only)

Use this notebook when you stopped training early, or when training is finished and you only want the exports. It does **not** train anything. It uses the `best.pt` files already saved on Drive, so the character detector may be a partly trained model if you interrupted stage 2. That is fine for a first test in the app.

Before you start:
1. Runtime menu, Change runtime type, choose a GPU (T4 is fine).
2. Drive must already contain:
   - `MyDrive/bd-plate-reader/outputs/plate_detector/weights/best.pt`
   - `MyDrive/bd-plate-reader/outputs/char_detector/weights/best.pt`
   - the dataset archives in `MyDrive/bd-plate-reader/data/` (the same ones used for training, needed to rebuild the calibration images and the test sets).
3. Run the cells from top to bottom.

Results are saved to `MyDrive/bd-plate-reader/outputs/models/` and `MyDrive/bd-plate-reader/outputs/eval_results.json`.

In [ ]:
# Settings. Change only if you know why.
REPO_URL = "https://github.com/Kanto065/bd-plate-reader.git"
BRANCH = "finish-notebook"    # use "main" once this branch is merged

In [ ]:
# Connect Drive and check the GPU.
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi -L
!ls -lh "/content/drive/MyDrive/bd-plate-reader/data"

In [ ]:
# Get the code and install requirements.
import os
if not os.path.isdir("/content/bd-plate-reader"):
    !git clone -b {BRANCH} {REPO_URL} /content/bd-plate-reader
else:
    !git -C /content/bd-plate-reader pull
%cd /content/bd-plate-reader/training
!pip install -q -r requirements.txt

## Prepare the data
Unpacks the archives and rebuilds the datasets. Export needs the validation images for int8 calibration, and evaluation needs the test sets. This does not retrain anything and takes a few minutes.

In [ ]:
!python organize_data.py
!python merge_datasets.py

## Check the saved weights
Prints the size and last modified time of both `best.pt` files, and the last epoch recorded in each `results.csv`. If the character detector shows fewer epochs than you expected, that is the point where training stopped.

In [ ]:
import csv, pathlib, time

base = pathlib.Path("/content/drive/MyDrive/bd-plate-reader/outputs")
missing = []
for name in ("plate_detector", "char_detector"):
    w = base / name / "weights" / "best.pt"
    if not w.exists():
        missing.append(str(w))
        print("MISSING", w)
        continue
    st = w.stat()
    print("%s: best.pt %.1f MB, saved %s" % (
        name, st.st_size / 1e6, time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(st.st_mtime))))
    res = base / name / "results.csv"
    if res.exists():
        rows = list(csv.reader(open(res, encoding="utf-8")))
        head = [h.strip() for h in rows[0]]
        last = rows[-1]
        get = lambda key: last[head.index(key)].strip() if key in head else "?"
        print("   last epoch logged: %s, mAP50 %s, mAP50-95 %s" % (
            get("epoch"), get("metrics/mAP50(B)"), get("metrics/mAP50-95(B)")))
    else:
        print("   no results.csv found")
if missing:
    raise RuntimeError("Missing weights, cannot continue: " + ", ".join(missing))
print("\nBoth best.pt files found. Continue to the export cell.")

## Export to int8 TFLite
Uses the existing `best.pt` of each stage. The first export installs TensorFlow packages and can take a few minutes.

In [ ]:
!python export.py

## Evaluate on the test sets
Prints mAP for both models and the plate level exact match on the LPDB-A test crops and on the held out DatasetNinja photos, reported separately.

In [ ]:
!python evaluate.py

In [ ]:
# What was saved to Drive.
!ls -lh /content/drive/MyDrive/bd-plate-reader/outputs/models
!cat /content/drive/MyDrive/bd-plate-reader/outputs/eval_results.json
print()
print("NEXT STEPS")
print("1. Open this Drive folder: My Drive / bd-plate-reader / outputs / models")
print("2. Download plate_detector_int8.tflite and char_detector_int8.tflite")
print("3. Also download eval_results.json from My Drive / bd-plate-reader / outputs")
print("4. Give the three files to Claude, and tell Claude which folder they are in.")